# Exercise 08: Web Search

> **Chapter:** Ch08 - Web Search
> **Estimated time:** ~50 minutes
>
> This exercise is optional. No submission, no grading.
> The solution notebook is released one week after the exercise.

PageRank looks simple: a page is important if important pages link to it. But what does it do when
people change the link graph, by accident or on purpose? In this exercise you take a real link
graph of 4,604 Wikipedia articles and change it in seven ways. A spammer builds a link farm, a race
report appears, a brand changes its name, an editor hoards links, and a template adds one link to
every page. You first guess by gut feeling what each change does. Then you implement PageRank, run
every change, and find out where your intuition was wrong. A last part asks whether the result of
the link farm carries over to a web of billions of pages.

**Your tasks** (tick them off as you go):

- [ ] **Quiz** - 20 questions in the quiz app
- [ ] ✏ **T1: Seven changes, seven guesses** - predict by gut feeling how each change moves PageRank
- [ ] 💻 **C1: PageRank by power iteration** - implement PageRank for a link graph (in this notebook)
- [ ] ✏ **T2: Where your guesses failed** - speculate why, within a strict time box
- [ ] ✏ **T3: The farm on a growing web** - predict and explain how a link farm fares as the web grows

---

## Quiz

Open the quiz app and work through the **20 questions** for this chapter. On the start screen,
pick the topic **"08 - Web Search"**:

**Quiz app:** https://roger-weber.github.io/mmir-unibasel-hs26/quiz/

The quiz covers definitions and basic concepts. The tasks below go further: you watch PageRank
react to changes of a real link graph and confront your intuition with the result.

---

## The Wikipedia link graph

The graph is the Wikispeedia graph from Stanford's SNAP collection. It holds the 4,604 articles of
the 2007 Wikipedia for Schools selection and the 119,772 links between them. Each article is a
page, and each link from one article to another is a directed edge, just like a hyperlink on the
web. The selection covers countries, history, science, people, and culture, so the graph has
natural hubs and natural authorities.

We ran PageRank with $\alpha = 0.85$ on this graph. You will reproduce these numbers in C1. Since
the average page has a PageRank of $1/N$, we report every score as a **multiple of the average
page**: ×1 is an average page, ×10 a page that the random surfer visits ten times as often.

| rank | article | score | incoming links | outgoing links |
|---|---|---|---|---|
| 1 | United States | ×44.1 | 1,551 | 294 |
| 2 | France | ×29.7 | 959 | 85 |
| 3 | Europe | ×29.3 | 933 | 159 |
| 4 | United Kingdom | ×28.8 | 972 | 168 |
| 9 | Latin | ×20.3 | 443 | 29 |

Further down the list, rank 10 scores ×18.7, rank 100 ×6.0, rank 500 ×2.1, and rank 1,000 ×1.15.
The median article scores ×0.40. Nobody links to 474 of the articles. They only receive the random
jump, score ×0.15, and share the last place at rank 4,131. A new article without incoming links
starts there too.

---

## Seven changes to the graph

Each change below edits the graph and reruns PageRank. Read them all, then make your guesses in
T1.

1. **A link farm.** An author publishes a new article "Google Search is run by humans" and wants
   it at the top. They create 1,000 stub articles that each contain one link, to the fake article.
   (a) The fake article itself has no links. (b) The fake article links back to all 1,000 stubs.
2. **A race report.** After the race weekend, a new article "2026 Monaco Grand Prix" appears. The
   articles Formula One (rank 1,244), Monaco Grand Prix (rank 4,028), and Monaco (rank 493) add a
   link to it, and it links back to these three.
3. **A brand rename.** The company renames its article "Sony" (rank 940, ×1.21, 18 incoming links)
   to "Sony Group". The article keeps its text and its outgoing links. (a) The old title is
   deleted. Links to it now point to a missing page and count for nothing. (b) The old title stays
   as a redirect: a page with one link, to "Sony Group". No other article is changed yet. (c) The
   redirect stays, and over time the authors update every link from "Sony" to "Sony Group".
4. **Two pages for one topic.** (a) Someone creates a second article "Robert Zimmerman", Bob
   Dylan's civil name, with the same links as "Bob Dylan" (rank 1,378). Half of the articles that
   linked to "Bob Dylan" now link to "Robert Zimmerman" instead. The two articles link to each
   other. (b) A French version "Paris (French)" is added next to "Paris" (rank 40, ×10.29). The two
   link to each other, as language versions do.
5. **Hoarding links.** The editors of "United States" argue that PageRank is a probability
   distribution: the less their article passes on to others, the more it keeps for itself. (a) They
   remove all 294 outgoing links. (b) They keep only three: France, United Kingdom, and Europe.
6. **A template link.** The license article "Wikipedia Text of the GNU Free Documentation License"
   exists in the graph but has no links at all. (a) A page template adds a link to it at the
   bottom of every article. (b) In addition, the license article links to the article
   "Wikipedia" (rank 2,014).
7. **A crowded hub or a quiet page.** A new article "Rhine swimming in Basel" has no incoming
   links. Its author can buy a mention on "United States", which already links to 294 articles.
   Or they can get a link from "Duchenne muscular dystrophy", an article near the bottom (rank
   3,972, ×0.16) that has no other outgoing links.

> **How this works:** Write your answer in the markdown cell below the question (replace the empty
> cells of the table). The solution notebook fills the same cell with the actual results, so you
> can compare.

### ✏ Task T1 - Seven changes, seven guesses

Go with your gut feeling. Spend about **one minute per change**, and do not calculate. A rank, a
rough score, or "up", "down", or "unchanged" is enough.

| change | question | your guess |
|---|---|---|
| 1a | rank of the fake article, farm only | |
| 1b | rank and score of the fake article, with back-links | |
| 2 | rank of the race report, and is it above or below "Monaco Grand Prix"? | |
| 3a | rank of "Sony Group", old title deleted | |
| 3b | rank of "Sony Group", with the redirect | |
| 3c | rank of "Sony Group", redirect and all links updated | |
| 4a | ranks of "Bob Dylan" and "Robert Zimmerman" | |
| 4b | what happens to "Paris"? | |
| 5a | score of "United States" without outgoing links | |
| 5b | score of "United States" with three outgoing links | |
| 5 | what happens to the 294 articles it used to link to? | |
| 6a | rank of the license article, and what happens to "United States"? | |
| 6b | rank of "Wikipedia" | |
| 7 | how many links from quiet pages like "Duchenne muscular dystrophy" equal one link from "United States"? | |

---

## Setup

In [ ]:
# Standard setup - run this first
import sys, pathlib, time

# Make `shared/` importable regardless of notebook depth
# (exercise notebooks live in exercises/chNN/, solutions in exercises/chNN/solution/).
_p = pathlib.Path().resolve()
while not (_p / "shared").is_dir() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

%load_ext autoreload
%autoreload 2

import numpy as np

from shared.display import display_md, print_table
from shared.wiki_graph import load_wiki_graph, synthetic_web, rank_of, times_average

WIKI = load_wiki_graph()
display_md(f"**{len(WIKI):,} articles** and **{sum(len(out) for out in WIKI.values()):,} links**. "
           f"'United States' links to {len(WIKI['United States'])} articles, for example "
           f"{', '.join(WIKI['United States'][:4])}.")

A graph is a dict that maps each page to the list of pages it links to. Every link target is also
a key, a page appears at most once in a list, and no page links to itself. `load_wiki_graph()`
returns a fresh copy on every call, so each change starts from the original graph.
`rank_of(pr, page)` returns the rank of a page in a PageRank result, and `times_average(pr, page)`
its score as a multiple of the average page.

---

## PageRank by power iteration

> **How this works:** Implement the function in the stub cell (replace `# YOUR CODE HERE`). Run
> the verify cell; the assertions either pass or fail with a specific message. Write the code
> yourself or direct an AI to implement it; the goal is to understand the result.

### 💻 Task C1 - PageRank by power iteration

Write `pagerank(graph, alpha, eps)` as defined in the chapter's section on link analysis:

- Every page starts with $1/N$.
- In each iteration, every page receives $\frac{1-\alpha}{N}$ from the random jump, plus $\alpha$
  times an equal share of the PageRank of every page that links to it.
- A page without outgoing links (a dangling page) sends the surfer to a random page: its
  PageRank is spread evenly over all $N$ pages.
- Stop when the L1 change between two iterations drops below `eps`, and return a dict that maps
  every page to its PageRank. The values sum to 1.
- Do not change `graph`.

Later cells run your function on graphs with up to 200,000 pages and 4 million links. Each
iteration should therefore visit every link only once.

In [ ]:
def pagerank(graph: dict, alpha: float = 0.85, eps: float = 1e-8) -> dict:
    """PageRank of every page of `graph` (page -> list of linked pages), by power iteration."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# --- Verify C1 ---
# The seven-page web of the chapter (figure 8.10).
BOOK = {"A": ["B", "E", "G"], "B": ["E"], "C": ["A", "B", "E"], "D": ["B", "C", "E", "G"],
        "E": ["C", "G"], "F": ["B", "D", "E", "G"], "G": ["E"]}
copy = {p: list(out) for p, out in BOOK.items()}
pr = pagerank(BOOK)
assert pr is not None, "pagerank returns None, did you forget `return`?"
assert BOOK == copy, "pagerank must not change the graph"
assert set(pr) == set(BOOK), "return a value for every page"
assert abs(sum(pr.values()) - 1) < 1e-6, f"the values must sum to 1, got {sum(pr.values()):.6f}"
for page, expected in {"E": 0.375, "G": 0.212, "C": 0.186, "B": 0.105, "F": 0.021}.items():
    assert abs(pr[page] - expected) < 0.001, f"PR({page}) = {pr[page]:.4f}, the chapter has {expected}"
got = pagerank(BOOK, alpha=0.5)["E"]
assert abs(got - 0.287) < 0.001, f"with alpha = 0.5, PR(E) = {got:.4f}, the chapter has 0.287"

pr = pagerank({"A": ["B"], "B": []})
assert abs(pr["B"] - 0.6491) < 1e-3, (f"A -> B with B dangling: expected A 0.3509, B 0.6491, "
                                      f"got A {pr['A']:.4f}, B {pr['B']:.4f}. Is B's PageRank spread over all pages?")
pr = pagerank({"A": [], "B": [], "C": []})
assert all(abs(v - 1 / 3) < 1e-6 for v in pr.values()), "a graph without links gives every page 1/3"

start = time.time()
BASE = pagerank(load_wiki_graph())
seconds = time.time() - start
assert abs(sum(BASE.values()) - 1) < 1e-6, f"the values must sum to 1, got {sum(BASE.values()):.6f}"
top = sorted(BASE, key=BASE.get, reverse=True)
assert top[:3] == ["United States", "France", "Europe"], f"the top 3 should be United States, France, Europe, got {top[:3]}"
assert abs(times_average(BASE, "United States") - 44.07) < 0.05, \
    f"United States should score x44.07, got x{times_average(BASE, 'United States'):.2f}"
assert rank_of(BASE, "Latin") == 9, f"Latin should rank 9th, got {rank_of(BASE, 'Latin')}"
assert seconds < 20, f"one run on the Wikipedia graph took {seconds:.0f} s: does each iteration visit every link only once?"
print(f"✓ pagerank correct! (Wikipedia graph in {seconds:.1f} s)")

In [ ]:
# --- Explore C1 ---
indegree = {p: 0 for p in WIKI}
for out in WIKI.values():
    for q in out:
        indegree[q] += 1
rows = [[rank_of(BASE, p), p, times_average(BASE, p), indegree[p], len(WIKI[p])] for p in top[:15]]
print_table(rows, headers=["Rank", "Article", "× average", "Incoming links", "Outgoing links"],
            floatfmt=("", "", ".2f", "", ""))
by_indegree = sorted(WIKI, key=indegree.get, reverse=True)
display_md(f"By incoming links, Latin is number **{by_indegree.index('Latin') + 1}**. "
           f"By PageRank, it is number **{rank_of(BASE, 'Latin')}**.")
# Which articles rank higher by PageRank than by their number of incoming links, and what do their linking pages have in common?

## What really happened

Each cell below builds one change of T1 and runs your `pagerank`. The helper `show` prints the
rank and the score of some pages before the change and in each variant. Compare the results with
your guesses in T1. The cells only need C1, so you can run them all at once.

In [ ]:
def show(variants: dict, pages: list) -> None:
    """Rank and score of `pages` before the change and in each variant {label: PageRank}."""
    rows = []
    for page in pages:
        for label, pr in {"before": BASE, **variants}.items():
            if page in pr:
                rows.append([page, label, rank_of(pr, page), times_average(pr, page)])
    print_table(rows, headers=["Page", "Graph", "Rank", "× average"], floatfmt=("", "", "", ".2f"))

**1. A link farm.**

In [ ]:
FAKE = "Google Search is run by humans"
STUBS = [f"Stub {i}" for i in range(1000)]

def farm(back_links: bool) -> dict:
    g = load_wiki_graph()
    g[FAKE] = list(STUBS) if back_links else []
    for stub in STUBS:
        g[stub] = [FAKE]
    return pagerank(g)

show({"(a) farm": farm(False), "(b) farm and back-links": farm(True)}, [FAKE, "United States"])

**2. A race report.**

In [ ]:
RACE = "2026 Monaco Grand Prix"
SOURCES = ["Formula One", "Monaco Grand Prix", "Monaco"]
g = load_wiki_graph()
for page in SOURCES:
    g[page].append(RACE)
g[RACE] = list(SOURCES)
show({"race report": pagerank(g)}, [RACE] + SOURCES)

**3. A brand rename.**

In [ ]:
def rename(redirect: bool, links_updated: bool) -> dict:
    g = load_wiki_graph()
    g["Sony Group"] = g.pop("Sony")        # the article keeps its own links
    for page, out in g.items():
        if "Sony" in out:
            if links_updated:
                g[page] = ["Sony Group" if t == "Sony" else t for t in out]
            elif not redirect:
                g[page] = [t for t in out if t != "Sony"]   # a link to a missing page counts for nothing
    if redirect:
        g["Sony"] = ["Sony Group"]
    return pagerank(g)

show({"(a) deleted": rename(False, False), "(b) redirect": rename(True, False),
      "(c) redirect, links updated": rename(True, True)}, ["Sony", "Sony Group"])

**4. Two pages for one topic.**

In [ ]:
g = load_wiki_graph()
linking = sorted(p for p, out in g.items() if "Bob Dylan" in out)
for page in linking[1::2]:                 # every second linking article switches
    g[page] = ["Robert Zimmerman" if t == "Bob Dylan" else t for t in g[page]]
g["Robert Zimmerman"] = g["Bob Dylan"] + ["Bob Dylan"]
g["Bob Dylan"] = g["Bob Dylan"] + ["Robert Zimmerman"]
show({"(a) split": pagerank(g)}, ["Bob Dylan", "Robert Zimmerman"])

g = load_wiki_graph()
g["Paris"].append("Paris (French)")
g["Paris (French)"] = ["Paris"]
show({"(b) French version": pagerank(g)}, ["Paris", "Paris (French)"])

**5. Hoarding links.**

In [ ]:
US = "United States"
TARGETS = WIKI[US]                         # the 294 articles it links to
variants = {}
for label, keep in [("(a) no outgoing links", []), ("(b) three outgoing links", ["France", "United Kingdom", "Europe"])]:
    g = load_wiki_graph()
    g[US] = keep
    variants[label] = pagerank(g)
show(variants, [US, "France"])

rows = []
for label, pr in variants.items():
    change = {t: pr[t] / BASE[t] - 1 for t in TARGETS}
    worst = min(change, key=change.get)
    rows.append([label, f"{np.median(list(change.values())):+.0%}", worst, f"{change[worst]:+.0%}",
                 f"{rank_of(BASE, worst)} -> {rank_of(pr, worst)}"])
print_table(rows, headers=["Graph", "Median change of the 294 former targets", "Largest loss", "Change", "Rank"])

**6. A template link.**

In [ ]:
LICENSE = "Wikipedia Text of the GNU Free Documentation License"

def template(license_links: list) -> dict:
    g = load_wiki_graph()
    for page, out in g.items():
        if page != LICENSE:
            out.append(LICENSE)
    g[LICENSE] = license_links
    return pagerank(g)

on_every_page = template([])
show({"(a) template link": on_every_page, "(b) license links to Wikipedia": template(["Wikipedia"])},
     [LICENSE, US, "Wikipedia"])

order = sorted(BASE, key=BASE.get, reverse=True)
rows = []
for lo, hi in [(1, 10), (11, 100), (101, 1000), (1001, 2500), (2501, len(order))]:
    pages = [p for p in order[lo - 1:hi] if p != LICENSE]
    rows.append([f"{lo:,} to {hi:,}", f"{np.mean([on_every_page[p] / BASE[p] for p in pages]) - 1:+.1%}"])
print_table(rows, headers=["Rank before", "Mean change in (a)"])

**7. A crowded hub or a quiet page.**

In [ ]:
NEW = "Rhine swimming in Basel"
rows = []
for source in ["United States", "Duchenne muscular dystrophy", "Latin"]:
    g = load_wiki_graph()
    g[NEW] = []
    g[source] = g[source] + [NEW]
    pr = pagerank(g)
    rows.append([source, rank_of(BASE, source), times_average(BASE, source), len(WIKI[source]),
                 rank_of(pr, NEW), times_average(pr, NEW)])
print_table(rows, headers=["Link from", "Its rank", "Its score", "Its other links", "Rank of the new article", "Score of the new article"],
            floatfmt=("", "", ".2f", "", "", ".2f"))
display_md(f"Without any incoming link, the new article starts at rank {rank_of(BASE, top[-1]):,} with ×0.15.")

> **How this works:** Write your answer in the markdown cell below the question (replace
> *Your answer here...*). The solution notebook fills the same cell with a model answer, so you
> can compare and reflect.

### ✏ Task T2 - Where your guesses failed

Compare the results with your guesses in T1. Pick the **two or three changes** where your guess
was furthest off. For each one, speculate in two or three sentences why PageRank behaves this way.
Think about where the score of a page comes from and where it goes. Set yourself a time box of
**15 minutes**. Speculating is fine: the solution explains all seven changes in full.

> *Your answer here...*

---

## The farm on a growing web

The link farm of change 1 took rank 1 in a graph of 4,604 articles. The chapter says the opposite
about the web: "A link farm of a thousand pages that nobody links to has a combined PageRank of
only about $1000 \cdot \frac{1-\alpha}{N}$, a negligible amount on a web of billions of pages." To
see what the size of the graph does, we use synthetic webs. `synthetic_web(n)` builds a web of `n`
pages in which every page has about 20 outgoing links. Each link picks its target by popularity:
page $i$ is chosen with a probability proportional to $(i+1)^{-0.9}$. A few pages collect most of
the links and many pages collect almost none, as on the real web. The webs have 5,000, 50,000,
and 200,000 pages, and the same 1,000-page farm as in change 1a is added to each.

### ✏ Task T3 - The farm on a growing web

First, **before you run the next cell**, write down your gut feeling: where does the fake article
rank in each of the three webs? Then run the cell, which takes up to a minute. Come back and
explain the result in three to five sentences. How does it fit both the result of change 1 and the
sentence of the chapter? What would the same farm achieve on the real web of hundreds of billions
of pages?

> *Your guess: ...*
>
> *Your explanation after running the cell: ...*

In [ ]:
rows = []
for n in [5_000, 50_000, 200_000]:
    web = synthetic_web(n)
    links = sum(len(out) for out in web.values())
    scores = sorted(pagerank(web).values(), reverse=True)
    web[FAKE] = []
    for stub in STUBS:
        web[stub] = [FAKE]
    pr = pagerank(web)
    rank = rank_of(pr, FAKE)
    rows.append([f"{n:,}", f"{links:,}", scores[0] * n, scores[99] * n, rank, f"{rank / len(pr):.3%}",
                 times_average(pr, FAKE)])
print_table(rows, headers=["Pages", "Links", "Top page", "Rank 100", "Rank of the fake article",
                           "Within the top", "Score of the fake article"],
            floatfmt=("", "", ".0f", ".1f", "", "", ".0f"))